# torch.fx — Graph-Level Model Transformation

**Module 23** of the PyTorch Complete Learning Guide

In this notebook, we explore `torch.fx` — PyTorch's Python-to-Python transformation framework.
We'll learn to capture models into a graph IR, inspect the graph, write transformation passes,
and use pattern matching to rewrite models automatically.

**What you'll learn:**
- Symbolic tracing and the FX Graph IR
- Iterating and inspecting graph nodes
- Writing graph transformation passes
- Pattern matching with `replace_pattern`
- ShapeProp for shape analysis
- `Interpreter` and `Transformer` for custom execution/rewriting
- Where FX fits in `torch.compile`

In [ ]:
import torch
import torch.nn as nn
import torch.fx
import time
from collections import Counter

print(f"PyTorch version: {torch.__version__}")

## 1. What is torch.fx?

`torch.fx` lets you:
1. **Capture** a module's forward logic into a graph IR
2. **Inspect** and **modify** that graph
3. **Generate** a new executable module from the modified graph

It's used internally by `torch.compile`, quantization, and distributed optimizations.

## 2. Symbolic Tracing a Model

In [ ]:
class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear1 = nn.Linear(10, 20)
        self.bn = nn.BatchNorm1d(20)
        self.linear2 = nn.Linear(20, 5)

    def forward(self, x):
        x = self.linear1(x)
        x = self.bn(x)
        x = torch.relu(x)
        x = self.linear2(x)
        return x

model = SimpleModel()
traced = torch.fx.symbolic_trace(model)

print("Type:", type(traced))
print("Is nn.Module:", isinstance(traced, nn.Module))
print("\n--- Generated Code ---")
print(traced.code)

In [ ]:
# Verify the traced module produces the same output
x = torch.randn(4, 10)
model.eval()
traced.eval()

out_orig = model(x)
out_traced = traced(x)
print(f"Outputs match: {torch.allclose(out_orig, out_traced)}")
print(f"Output shape: {out_traced.shape}")

## 3. The Graph — Tabular View

In [ ]:
traced.graph.print_tabular()

In [ ]:
# String representation of the graph
print(traced.graph)

## 4. Understanding Nodes — Iterate and Inspect

Each node has:
- **op**: placeholder, get_attr, call_function, call_method, call_module, output
- **name**: unique identifier
- **target**: what to call
- **args/kwargs**: inputs
- **users**: downstream consumers

In [ ]:
for node in traced.graph.nodes:
    print(f"op={node.op:15s} name={node.name:15s} target={str(node.target):30s}")
    print(f"  args={node.args}")
    print(f"  users={[u.name for u in node.users]}")
    print()

In [ ]:
# Count operations by type
op_counts = Counter()
for node in traced.graph.nodes:
    if node.op == "call_function":
        op_counts[f"fn:{node.target.__name__}"] += 1
    elif node.op == "call_module":
        mod = traced.get_submodule(node.target)
        op_counts[f"mod:{type(mod).__name__}"] += 1
    elif node.op == "call_method":
        op_counts[f"method:{node.target}"] += 1

print("Operation counts:")
for op, count in op_counts.most_common():
    print(f"  {op}: {count}")

## 5. Graph Transformation — Replace ReLU with GELU

In [ ]:
class ReLUModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear1 = nn.Linear(10, 20)
        self.relu1 = nn.ReLU()
        self.linear2 = nn.Linear(20, 20)
        self.linear3 = nn.Linear(20, 5)

    def forward(self, x):
        x = self.relu1(self.linear1(x))
        x = torch.relu(self.linear2(x))
        x = self.linear3(x)
        return x

relu_model = ReLUModel()
relu_traced = torch.fx.symbolic_trace(relu_model)
print("--- Before ---")
print(relu_traced.code)

In [ ]:
def replace_relu_with_gelu(gm):
    for node in gm.graph.nodes:
        if node.op == "call_function" and node.target in (
            torch.relu, torch.nn.functional.relu
        ):
            node.target = torch.nn.functional.gelu
        elif node.op == "call_module":
            mod = gm.get_submodule(node.target)
            if isinstance(mod, nn.ReLU):
                parent_name, _, attr_name = node.target.rpartition(".")
                parent = gm.get_submodule(parent_name) if parent_name else gm
                setattr(parent, attr_name, nn.GELU())
    gm.graph.lint()
    gm.recompile()
    return gm

replace_relu_with_gelu(relu_traced)
print("--- After ---")
print(relu_traced.code)

x = torch.randn(4, 10)
print(f"Output shape: {relu_traced(x).shape}")

## 6. Graph Transformation — Add Timing

In [ ]:
class ProfilingInterpreter(torch.fx.Interpreter):
    def __init__(self, module):
        super().__init__(module)
        self.node_profiles = {}

    def run_node(self, node):
        start = time.perf_counter()
        result = super().run_node(node)
        elapsed = time.perf_counter() - start
        self.node_profiles[node.name] = {
            "op": node.op,
            "time_ms": elapsed * 1000,
            "shape": result.shape if isinstance(result, torch.Tensor) else None,
        }
        return result

model = SimpleModel()
model.eval()
traced_for_profile = torch.fx.symbolic_trace(model)
profiler = ProfilingInterpreter(traced_for_profile)
output = profiler.run(torch.randn(32, 10))

print(f"{'Node':20s} {'Op':15s} {'Time (ms)':>10s} {'Shape':>20s}")
print("-" * 68)
for name, info in profiler.node_profiles.items():
    shape_str = str(info['shape']) if info['shape'] else ''
    print(f"{name:20s} {info['op']:15s} {info['time_ms']:10.4f} {shape_str:>20s}")

## 7. Pattern Matching with replace_pattern

In [ ]:
from torch.fx import subgraph_rewriter

class PatternModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear1 = nn.Linear(10, 20)
        self.linear2 = nn.Linear(20, 5)

    def forward(self, x):
        x = self.linear1(x)
        x = torch.add(x, x)  # double
        x = torch.relu(x)
        x = self.linear2(x)
        return x

pat_model = PatternModel()
pat_traced = torch.fx.symbolic_trace(pat_model)
print("--- Before ---")
print(pat_traced.code)

In [ ]:
def pattern(x):
    a = torch.add(x, x)
    b = torch.relu(a)
    return b

def replacement(x):
    return torch.nn.functional.gelu(torch.mul(x, 2.0))

matches = subgraph_rewriter.replace_pattern(pat_traced, pattern, replacement)
pat_traced.graph.lint()
pat_traced.recompile()

print(f"Replaced {len(matches)} match(es)")
print("\n--- After ---")
print(pat_traced.code)

x = torch.randn(4, 10)
print(f"Output shape: {pat_traced(x).shape}")

## 8. ShapeProp — Shape Analysis

In [ ]:
from torch.fx.passes.shape_prop import ShapeProp

model = SimpleModel()
model.eval()
traced = torch.fx.symbolic_trace(model)

sample = torch.randn(8, 10)
ShapeProp(traced).propagate(sample)

print(f"{'Node':15s} {'Shape':25s} {'Dtype':15s}")
print("-" * 58)
for node in traced.graph.nodes:
    meta = node.meta.get("tensor_meta")
    if meta is not None and not isinstance(meta, tuple):
        print(f"{node.name:15s} {str(meta.shape):25s} {str(meta.dtype):15s}")
    else:
        print(f"{node.name:15s} {'(no tensor meta)':25s}")

## 9. FX Interpreter — Custom Execution

The `Interpreter` runs a graph node-by-node, letting you override behavior at each step.

In [ ]:
class ShapeTrackingInterpreter(torch.fx.Interpreter):
    def __init__(self, module):
        super().__init__(module)
        self.shapes = {}

    def run_node(self, node):
        result = super().run_node(node)
        if isinstance(result, torch.Tensor):
            self.shapes[node.name] = {
                "shape": result.shape,
                "dtype": result.dtype,
                "numel": result.numel(),
            }
        return result

model = SimpleModel()
model.eval()
traced = torch.fx.symbolic_trace(model)

tracker = ShapeTrackingInterpreter(traced)
output = tracker.run(torch.randn(4, 10))

for name, info in tracker.shapes.items():
    print(f"{name:15s} shape={str(info['shape']):20s} numel={info['numel']}")

## 10. FX Transformer — Clean Node-Level Transforms

In [ ]:
class AddClampTransformer(torch.fx.Transformer):
    """Add clamping after every linear layer."""
    def call_module(self, target, args, kwargs):
        result = super().call_module(target, args, kwargs)
        mod = self.fetch_attr(target)
        if isinstance(mod, nn.Linear):
            result = super().call_function(
                torch.clamp, (result,), {"min": -10.0, "max": 10.0}
            )
        return result

model = SimpleModel()
model.eval()
traced = torch.fx.symbolic_trace(model)

print("--- Before ---")
print(traced.code)

transformed = AddClampTransformer(traced).transform()
print("--- After ---")
print(transformed.code)

x_large = torch.randn(4, 10) * 100
out = transformed(x_large)
print(f"Max abs value: {out.abs().max().item():.1f} (clamped)")

## 11. Dead Code Elimination

In [ ]:
class DeadCodeModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear1 = nn.Linear(10, 20)
        self.linear2 = nn.Linear(10, 20)
        self.linear3 = nn.Linear(20, 5)

    def forward(self, x):
        a = self.linear1(x)
        _unused = self.linear2(x)  # dead code
        out = self.linear3(a)
        return out

dc_model = DeadCodeModel()
dc_traced = torch.fx.symbolic_trace(dc_model)

before_count = len(list(dc_traced.graph.nodes))
print(f"Before DCE: {before_count} nodes")
print(dc_traced.code)

dc_traced.graph.eliminate_dead_code()
dc_traced.recompile()

after_count = len(list(dc_traced.graph.nodes))
print(f"After DCE: {after_count} nodes (removed {before_count - after_count})")
print(dc_traced.code)

## 12. Where FX Fits in torch.compile

```
torch.compile(model)
      │
      ▼
TorchDynamo  ─── Captures Python bytecode → FX Graph
      │
      ▼
AOTAutograd  ─── Joint fwd+bwd → partitions into two FX Graphs
      │
      ▼
Inductor     ─── Lowers FX Graph → Triton/C++ code
```

You can write a custom backend that receives the FX graph:

In [ ]:
def debug_backend(gm: torch.fx.GraphModule, example_inputs):
    """Custom backend that prints the graph from Dynamo."""
    print("--- FX graph from Dynamo ---")
    for node in gm.graph.nodes:
        print(f"  {node.op:15s} {node.name}")
    print(f"  Total nodes: {len(list(gm.graph.nodes))}")
    return gm

model = SimpleModel()
model.eval()
compiled = torch.compile(model, backend=debug_backend)
compiled(torch.randn(4, 10))
print("\nDynamo produces more granular ATen-level ops than symbolic_trace.")

## 13. Fuse Consecutive Linear Layers

In [ ]:
class TwoLinear(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear1 = nn.Linear(10, 20)
        self.linear2 = nn.Linear(20, 5)

    def forward(self, x):
        return self.linear2(self.linear1(x))

fuse_model = TwoLinear()
fuse_traced = torch.fx.symbolic_trace(fuse_model)
print("Before fusion:")
print(fuse_traced.code)

x = torch.randn(4, 10)
out_before = fuse_model(x)

# Fuse
for node in list(fuse_traced.graph.nodes):
    if node.op != "call_module":
        continue
    mod1 = fuse_traced.get_submodule(node.target)
    if not isinstance(mod1, nn.Linear):
        continue
    users = list(node.users.keys())
    if len(users) != 1 or users[0].op != "call_module":
        continue
    next_node = users[0]
    mod2 = fuse_traced.get_submodule(next_node.target)
    if not isinstance(mod2, nn.Linear):
        continue

    with torch.no_grad():
        W = mod2.weight @ mod1.weight
        b = mod2.weight @ mod1.bias + mod2.bias
    fused = nn.Linear(mod1.in_features, mod2.out_features)
    with torch.no_grad():
        fused.weight.copy_(W)
        fused.bias.copy_(b)

    fuse_traced.add_module("fused", fused)
    with fuse_traced.graph.inserting_before(node):
        fn = fuse_traced.graph.call_module("fused", args=node.args)
    next_node.replace_all_uses_with(fn)
    fuse_traced.graph.erase_node(next_node)
    fuse_traced.graph.erase_node(node)
    break

fuse_traced.graph.lint()
fuse_traced.recompile()

print("After fusion:")
print(fuse_traced.code)

out_after = fuse_traced(x)
print(f"Outputs match: {torch.allclose(out_before, out_after, atol=1e-5)}")
print(f"Max diff: {(out_before - out_after).abs().max().item():.2e}")

## 14. graph.lint() — Validate Your Graph

In [ ]:
model = SimpleModel()
model.eval()
traced = torch.fx.symbolic_trace(model)

try:
    traced.graph.lint()
    print("graph.lint() passed — graph is valid")
except Exception as e:
    print(f"Lint error: {e}")

# Try erasing a node that still has users
for node in traced.graph.nodes:
    if node.op == "call_function" and node.target == torch.relu:
        try:
            traced.graph.erase_node(node)
        except RuntimeError as e:
            print(f"\nCan't erase node with users: {e}")
        break

## 15. Exercise: Write an FX Pass That Counts FLOPs

**Goal:** Write an FX pass that estimates the number of floating-point operations (FLOPs) at each node.

**Hints:**
- For `nn.Linear(in, out)` with input batch B: FLOPs ≈ 2 × B × in × out
- For element-wise ops (relu, gelu, add): FLOPs ≈ numel
- Use ShapeProp to get tensor shapes at each node
- Subclass `Interpreter` and override `run_node`

In [ ]:
# YOUR CODE HERE
class FLOPCounter(torch.fx.Interpreter):
    def __init__(self, module):
        super().__init__(module)
        self.flop_counts = {}

    def run_node(self, node):
        result = super().run_node(node)

        flops = 0
        if node.op == "call_module":
            mod = self.fetch_attr(node.target)
            if isinstance(mod, nn.Linear) and isinstance(result, torch.Tensor):
                batch = result.shape[0]
                flops = 2 * batch * mod.in_features * mod.out_features
            elif isinstance(mod, nn.BatchNorm1d) and isinstance(result, torch.Tensor):
                flops = result.numel() * 5
        elif node.op in ("call_function", "call_method"):
            if isinstance(result, torch.Tensor):
                flops = result.numel()

        self.flop_counts[node.name] = flops
        return result

model = SimpleModel()
model.eval()
traced = torch.fx.symbolic_trace(model)

counter = FLOPCounter(traced)
counter.run(torch.randn(8, 10))

print(f"{'Node':15s} {'FLOPs':>10s}")
print("-" * 28)
total = 0
for name, flops in counter.flop_counts.items():
    if flops > 0:
        print(f"{name:15s} {flops:10,d}")
        total += flops
print(f"{'TOTAL':15s} {total:10,d}")

## Key Takeaways

1. **`torch.fx.symbolic_trace`** captures a module's forward into a graph IR — pure Python, easy to inspect
2. **FX nodes** have 6 op types: `placeholder`, `get_attr`, `call_function`, `call_method`, `call_module`, `output`
3. **Graph transforms** use `inserting_before/after`, `erase_node`, and `replace_all_uses_with`
4. **`replace_pattern`** finds and replaces subgraph patterns automatically
5. **ShapeProp** propagates shapes through the graph for optimization decisions
6. **`Interpreter`** executes node-by-node with custom hooks (profiling, type checking)
7. **`Transformer`** creates a new graph via clean per-node rewriting
8. **Always `graph.lint()`** after modifying and **`gm.recompile()`** before executing
9. **torch.compile** uses FX throughout: Dynamo → AOTAutograd → Inductor all operate on FX graphs